# Global and Local Moran's I

Fits global Moran's I and Local Moran's I (LISA) with Queen-contiguity spatial weights for the original dataset and every masked variant, classifies cells into HH/LH/LL/HL/NS clusters, and suppresses the LISA classification for any cell that does not meet the shared publication rule (`K_MIN` cases, `PMIN` population) — applied identically to every dataset, including the original, so no map in the comparison can reveal a sparsely populated, potentially identifiable cell. Use the `lisa_pub` field (not `lisa_cls`) for QGIS map styling.

**Input:** the datasets listed in `DATASETS` below, plus the population grid.
**Output:** one shapefile per dataset plus a summary CSV in `outputs_global_local_moran/`.


In [ ]:
#Imports
import os
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

from libpysal.weights import Queen
from esda.moran import Moran, Moran_Local


In [ ]:
#Paths
# Data directory. Override with the DATA_DIR environment variable to
# point at a different copy of the data without editing this file, e.g.
#   DATA_DIR=/path/to/data jupyter execute this_notebook.ipynb
# The data itself is not part of this repository -- see README.md.
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))

GRID_PATH = DATA_DIR / "population_grid" / "GHS_Hungary_pop_city-clipped2.shp"
GRID_ID_COL = "GID"
POP_COL = "VALUE"
PATIENT_ID_COL = None

#Parameters
TARGET_EPSG = 23700
N_PERMUTATIONS = 999
P_THRESHOLD = 0.05
RANDOM_SEED = 42

# Set to 0 to analyse the full grid
MIN_POP_FOR_ANALYSIS = 0

# Privacy safeguard for mapping: same publication thresholds used in
# Descriptive-Mapping.ipynb and Locally_Weighted_Average_Smoothing.ipynb.
# A cell's LISA classification is only released for mapping if it has at
# least K_MIN cases AND at least PMIN population - applied identically to
# every dataset, including the original (unmasked) data, so no map in the
# comparison can reveal sparsely populated, potentially identifiable cells.
K_MIN = 5
PMIN = 50

OUT_DIR = Path("outputs_global_local_moran")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
#Loading Datasets

DATASETS = {
    "original_points": {
        "type": "points",
        "path": DATA_DIR / "original" / "2008_eov_illness_stand.shp",
    },
    "donut": {
        "type": "points",
        "path": DATA_DIR / "donut_masking" / "donut_points150_high900.shp",
    },
    "stat_masked": {
        "type": "points",
        "path": DATA_DIR / "statistical_masking" / "statmasked_points.shp",
    },
    "synthetic_sdv": {
        "type": "points",
        "path": DATA_DIR / "synthetic" / "synthetic_points.shp",
    },
    "areal_grid_release": {
        "type": "grid",
        "path": DATA_DIR / "area_aggregation" / "aggregated_points.shp",
        "count_col": "n_cases_pu",
    },
}

In [ ]:
#functions

# For point-based datasets, we need to count the number of cases per grid cell using a spatial join.
def count_points_to_grid(points: gpd.GeoDataFrame, grid: gpd.GeoDataFrame) -> pd.Series:
    """Count point cases per grid cell."""
    if points.crs != grid.crs:
        points = points.to_crs(grid.crs)

    joined = gpd.sjoin(
        points[["geometry"] + ([PATIENT_ID_COL] if PATIENT_ID_COL else [])],
        grid[[GRID_ID_COL, "geometry"]],
        how="inner",
        predicate="within",
    )

    if PATIENT_ID_COL:
        counts = joined.groupby(GRID_ID_COL)[PATIENT_ID_COL].nunique()
    else:
        counts = joined.groupby(GRID_ID_COL).size()

    return counts.rename("Y")

# For grid-based datasets, we assume the counts are already aggregated and just read them in.
def read_grid_counts(masked_grid: gpd.GeoDataFrame, grid: gpd.GeoDataFrame, count_col: str) -> pd.Series:
    """Read already aggregated case counts from a grid layer."""
    if masked_grid.crs != grid.crs:
        masked_grid = masked_grid.to_crs(grid.crs)

    masked_grid[count_col] = pd.to_numeric(masked_grid[count_col], errors="coerce").fillna(0)
    return masked_grid.groupby(GRID_ID_COL)[count_col].sum().rename("Y")

# Function to classify Local Moran quadrants for later mapping
def lisa_clusters(lisa: Moran_Local, p_threshold: float = 0.05) -> np.ndarray:
    """Classify Local Moran quadrants using the default PySAL scheme:
    1=HH, 2=LH, 3=LL, 4=HL. Non-significant cells are labelled NS.
    """
    labels = np.array(["NS"] * len(lisa.q), dtype=object)
    sig = lisa.p_sim < p_threshold
    labels[(lisa.q == 1) & sig] = "HH"
    labels[(lisa.q == 2) & sig] = "LH"
    labels[(lisa.q == 3) & sig] = "LL"
    labels[(lisa.q == 4) & sig] = "HL"
    return labels

#load grid and prepare spatial weights

grid = gpd.read_file(GRID_PATH)

if TARGET_EPSG is not None:
    grid = grid.to_crs(epsg=TARGET_EPSG)

grid[POP_COL] = pd.to_numeric(grid[POP_COL], errors="coerce").fillna(0)
grid = grid[[GRID_ID_COL, POP_COL, "geometry"]].copy()

if MIN_POP_FOR_ANALYSIS > 0:
    grid = grid.loc[grid[POP_COL] >= MIN_POP_FOR_ANALYSIS].copy()

grid = grid.sort_values(GRID_ID_COL).reset_index(drop=True)

#Fitting spatial weights with Queen contiguity and row-standardization for later Moran's I analysis.
w = Queen.from_dataframe(grid, use_index=False)
w.transform = "r"

print(f"Analysis cells: {len(grid)}")
print(f"Islands without neighbours: {len(w.islands)}")

summary_rows = []

#Iterating through datasets, fitting global Moran's I and Local Moran's I, and collecting results for export and summary.
for dataset_name, spec in DATASETS.items():
    print(f"Processing {dataset_name}...")

    analysis_grid = grid.copy()

    if spec["type"] == "points":
        points = gpd.read_file(spec["path"])
        counts = count_points_to_grid(points, grid)

    elif spec["type"] == "grid":
        masked_grid = gpd.read_file(spec["path"])
        counts = read_grid_counts(masked_grid, grid, spec["count_col"])

    else:
        raise ValueError(f"Unknown dataset type: {spec['type']}")

    # Merging counts with the grid for Moran's I analysis and later mapping.
    analysis_grid = analysis_grid.merge(
        counts,
        left_on=GRID_ID_COL,
        right_index=True,
        how="left",
    )

    # Fill missing counts with 0 and ensure integer type for later QGIS mapping.
    analysis_grid["Y"] = analysis_grid["Y"].fillna(0).astype(int)

    y = analysis_grid["Y"].to_numpy(dtype=float)

    if np.std(y) == 0:
        print(f"Skipped {dataset_name}: Y has zero variance.")
        continue

    #Fitting global Moran's I and Local Moran's I with permutation tests.
    moran = Moran(
        y,
        w,
        transformation="r",
        permutations=N_PERMUTATIONS,
        two_tailed=True,
    )

    #Fitting Local Moran's I with the same parameters for later mapping.
    lisa = Moran_Local(
        y,
        w,
        transformation="r",
        permutations=N_PERMUTATIONS,
        seed=RANDOM_SEED,
    )

    #Adding results to the grid for later export and mapping
    analysis_grid["lisa_I"] = lisa.Is
    analysis_grid["lisa_p"] = lisa.p_sim
    analysis_grid["lisa_q"] = lisa.q
    analysis_grid["lisa_sig"] = (lisa.p_sim < P_THRESHOLD).astype(int)
    analysis_grid["lisa_cls"] = lisa_clusters(lisa, P_THRESHOLD)

    # --------------------------------------------------
    # Privacy safeguard: suppress the LISA classification for cells that
    # do not meet the same publication thresholds (K_MIN cases, PMIN
    # population) used for the choropleth and smoothed disease maps.
    # Applied identically to every dataset, including the original
    # (unmasked) data, so that no map in the comparison discloses the
    # classification of sparsely populated, potentially identifiable
    # cells. "lisa_cls" keeps the raw classification for internal QA;
    # "lisa_pub" is the field to use for QGIS map styling.
    # --------------------------------------------------
    analysis_grid["publish"] = (
        (analysis_grid["Y"] >= K_MIN) & (analysis_grid[POP_COL] >= PMIN)
    ).astype(int)
    analysis_grid["lisa_pub"] = np.where(
        analysis_grid["publish"] == 1,
        analysis_grid["lisa_cls"],
        "suppressed",
    )

    #Exporting one layer per dataset for QGIS map design.
    analysis_grid.to_file(OUT_DIR / f"{dataset_name}_local_global.shp", driver="ESRI Shapefile")

    #Collecting summary statistics for the global Moran's I and Local Moran's I results.
    n = len(analysis_grid)
    summary_rows.append({
        "dataset": dataset_name,
        "n_cells": n,
        "total_cases": int(analysis_grid["Y"].sum()),
        "mean_Y": float(analysis_grid["Y"].mean()),
        "var_Y": float(analysis_grid["Y"].var()),
        "moran_I": float(moran.I),
        "expected_I": float(moran.EI),
        "z_sim": float(moran.z_sim),
        "p_sim": float(moran.p_sim),
        "n_lisa_sig": int(analysis_grid["lisa_sig"].sum()),
        "pct_lisa_sig": float(analysis_grid["lisa_sig"].mean() * 100),
        "n_HH": int((analysis_grid["lisa_cls"] == "HH").sum()),
        "n_LL": int((analysis_grid["lisa_cls"] == "LL").sum()),
        "n_HL": int((analysis_grid["lisa_cls"] == "HL").sum()),
        "n_LH": int((analysis_grid["lisa_cls"] == "LH").sum()),
        "n_publishable_cells": int(analysis_grid["publish"].sum()),
        "n_suppressed_cells": int((analysis_grid["publish"] == 0).sum()),
    })

#Creating a summary table of results and saving it as CSV for later use in the manuscript and QGIS map labels
summary = pd.DataFrame(summary_rows).sort_values("dataset")
summary.to_csv(OUT_DIR / "summary_global_local_moran.csv", index=False)

print(summary)
print(f"QGIS layers saved to: {OUT_DIR / 'local_moran_layers.shp'}")
print("For map styling in QGIS, use the 'lisa_pub' field (not 'lisa_cls') so that")
print("suppressed cells below the K_MIN/PMIN threshold are not colour-coded.")
print(f"Summary saved to: {OUT_DIR / 'summary_global_local_moran.csv'}")